# Проверка окружения

Этот ноутбук проверяет, что всё готово к курсу: версия Python, пакеты, ключ и один тестовый вызов модели. **Выполните ячейки сверху вниз** (Shift+Enter).

Вы готовы, если последняя ячейка напечатала ответ модели и строку `usage`. Здесь показан пример того, как выглядит успех, — у вас должно получиться так же.

### 1. Версия Python
Нужна 3.10 или новее.

In [1]:
import sys
v = sys.version_info
print(f'Python {v.major}.{v.minor}.{v.micro}')
assert v >= (3, 10), 'Нужен Python 3.10+. Обновите Python или пересоздайте venv.'
print('OK: версия подходит')

Python 3.11.9
OK: версия подходит


### 2. Пакеты
Ставятся командой `pip install requests pydantic python-dotenv`.

In [2]:
missing = []
for pkg in ('requests', 'pydantic', 'dotenv'):
    try:
        __import__(pkg)
    except ImportError:
        missing.append('python-dotenv' if pkg == 'dotenv' else pkg)
if missing:
    print('НЕ хватает:', ', '.join(missing))
    print('Установите:  pip install ' + ' '.join(missing))
else:
    print('OK: все пакеты на месте')

OK: все пакеты на месте


### 3. Ключ
Ноутбук ищет ключ в переменной окружения `OPENROUTER_API_KEY` или в файле `.env` рядом.

В Kaggle используйте **Add-ons → Secrets** и добавьте секрет с именем `OPENROUTER_API_KEY` (его подхватит `os.environ`).

In [3]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()  # подхватит .env из текущей папки, если он есть
except Exception:
    pass
key = os.getenv('OPENROUTER_API_KEY')
if not key:
    print('Ключ не найден. Проверьте, что .env лежит рядом и содержит строку')
    print('OPENROUTER_API_KEY=sk-or-v1-...   (без пробелов и кавычек)')
else:
    print('OK: ключ найден,', key[:12] + '...' + key[-4:])

OK: ключ найден, sk-or-v1-abc...wxyz


### 4. Тестовый вызов модели
Один запрос к дешёвой модели. Если он прошёл — трубка от вашего кода до модели работает целиком.

In [4]:
import requests
resp = requests.post(
    'https://openrouter.ai/api/v1/chat/completions',
    headers={'Authorization': f'Bearer {key}'},
    json={
        'model': 'openai/gpt-4o-mini',
        'messages': [{'role': 'user', 'content': 'Ответь одним словом: всё работает?'}],
        'max_tokens': 20,
    },
    timeout=60,
)
resp.raise_for_status()
data = resp.json()
print('Ответ модели:', data['choices'][0]['message']['content'])
print('usage:', data['usage'])
print()
print('=' * 40)
print('  ВСЁ ГОТОВО. Увидимся на первой лекции.')
print('=' * 40)

Ответ модели: Да.
usage: {'prompt_tokens': 16, 'completion_tokens': 2, 'total_tokens': 18, 'cost': 3.6e-06, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 3.6e-06, 'upstream_inference_prompt_cost': 2.4e-06, 'upstream_inference_completions_cost': 1.2e-06}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}

  ВСЁ ГОТОВО. Увидимся на первой лекции.


---
**Если что-то не так**, в ячейке будет ошибка. Частые случаи разобраны в памятке `pamyatka_setup.pdf` (раздел «Если застряли»). Не мучайтесь в одиночку — пишите в чат курса с текстом ошибки, для того эти три дня и нужны.